# 05 - Evaluation Metrics
**Unit 5: Accuracy, Catalog Coverage, and Recall**

Quantifying the recommendation quality through offline performance metrics.

## Metrics Implemented:
- **Accuracy**: For classification elements (Fitness_Goal prediction)
- **RMSE**: Root Mean Square Error for SVD matrix factorization
- **Precision@K & Recall@K**: Top-K recommendation tasks
- **F1-Score**: Balancing precision and recall
- **Catalog Coverage**: Proportion of items ever recommended
- **Diversity (Jaccard)**: Intra-list similarity among recommended items


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH = os.path.join(NOTEBOOK_DIR, 'data', 'merged_fitness_data.csv')

if not os.path.exists(DATA_PATH):
    print(f"ERROR: {DATA_PATH} not found. Run merge_datasets.py first.")
else:
    # Load data
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset with {len(df)} rows for evaluation.")
    
    # Setup features
    numeric_features = ['Age', 'Height', 'Weight']
    categorical_features = ['Gender', 'Chronic_Disease', 'Activity_Level', 'Dietary_Preference', 'Fitness_Goal']
    
    # Handle missing values
    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())
    for col in categorical_features:
        df[col] = df[col].astype(str).fillna('None')
    
    # Train-test split
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Training set: {len(train_df)} | Test set: {len(test_df)}")
    
    # Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    X_train = preprocessor.fit_transform(train_df[numeric_features + categorical_features])
    X_test = preprocessor.transform(test_df[numeric_features + categorical_features])
    
    # KNN Model
    knn = NearestNeighbors(n_neighbors=20, metric='cosine')
    knn.fit(X_train)
    print("KNN Model trained.")
    
    # SVD Model
    interaction_matrix = pd.crosstab(train_df.index, train_df['diet_recommendation'])
    svd = TruncatedSVD(n_components=min(10, interaction_matrix.shape[1]-1), random_state=42)
    latent_space = svd.fit_transform(interaction_matrix)
    reconstructed = svd.inverse_transform(latent_space)
    rmse = np.sqrt(mean_squared_error(interaction_matrix.values.flatten(), reconstructed.flatten()))
    print(f"SVD Reconstruction RMSE: {rmse:.4f}")
    
    # Precision@K and Recall@K
    def precision_at_k(recommended, actual, k):
        if len(recommended) > k:
            recommended = recommended[:k]
        return len(set(recommended) & set(actual)) / len(recommended) if recommended else 0
    
    def recall_at_k(recommended, actual, k):
        if len(recommended) > k:
            recommended = recommended[:k]
        return len(set(recommended) & set(actual)) / len(actual) if actual else 0
    
    K = 5
    precisions, recalls = [], []
    test_indices = list(test_df.index)[:100]
    
    for idx in test_indices:
        try:
            test_vec = X_test[list(test_df.index).index(idx)]
            _, ind = knn.kneighbors([test_vec])
            recommended = train_df.iloc[ind[0]]['diet_recommendation'].tolist()
            actual = [test_df.loc[idx, 'diet_recommendation']]
            precisions.append(precision_at_k(recommended, actual, K))
            recalls.append(recall_at_k(recommended, actual, K))
        except:
            pass
    
    print(f"Precision@{K}: {np.mean(precisions):.4f}")
    print(f"Recall@{K}: {np.mean(recalls):.4f}")
    
    # Catalog Coverage
    unique_diets = df['diet_recommendation'].nunique()
    recommended_items = set()
    for i in range(min(500, len(X_test))):
        _, ind = knn.kneighbors([X_test[i]])
        recommended_items.update(train_df.iloc[ind[0]]['diet_recommendation'].tolist())
    coverage = len(recommended_items) / unique_diets
    print(f"Catalog Coverage: {coverage:.2%}")
    
    # Classification Accuracy
    X_cls = preprocessor.fit_transform(df[numeric_features + categorical_features])
    y_cls = df['Fitness_Goal']
    X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_cls, y_train_cls)
    y_pred = rf.predict(X_test_cls)
    print(f"Accuracy: {accuracy_score(y_test_cls, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test_cls, y_pred, average='macro'):.4f}")
    
    print("\n✓ Evaluation framework complete!")